
---

## **Part 1 — Exploring Different Modalities / Representations of Network Traffic**

Synthetic network traffic generation is useful for many applications such as dataset augmentation, network testing, and resource management. Many existing generation methods treat traffic generation as either a time-series prediction task or an autoregressive modeling task. In these approaches, models are trained directly on structured representations of packets—one common example is **nPrint**, a tabular format that encodes packet header fields from PCAP traces into machine-learning-friendly numerical vectors.

In an **nPrint**, each row represents one packet in a trace. The entire nPrint file (a large CSV) represents all packets of that trace in sequential order. Header fields are encoded using one-hot–like binary indicators (e.g., `1`, `0`, or `01`), making them easy to feed into ML models.

However, general-purpose ML models—even advanced time-series architectures and transformers—often struggle to capture the *complex dependencies* present in real network traffic:

* **Local dependencies**: relationships among columns within a single row (i.e., dependencies among header fields of a single packet).
* **Global dependencies**: relationships across rows (i.e., the evolution of packets within the same flow).
  For example: if the first packet of a flow uses TCP, the second packet in that same flow should also be TCP; sequence numbers, flags, and flow identifiers evolve in structured ways over time.

While sequential ML models struggle with these multi-scale dependencies, **vision models** (e.g., diffusion models) excel at capturing both local and global structure when data is presented spatially—like an image. By converting nPrint traces into 2D PNG images, we can take advantage of the strong representational capabilities of image models and generate synthetic traffic using visual generative approaches.

---

### **Your Task for Part 1**

In this part of the assignment, you will:

1. **Inspect the raw nPrint files** (found in the `real_nprints` directory).
2. **Understand how each CSV row and column corresponds to packet-level metadata.**
3. **Follow the provided conversion pipeline** that transforms these nPrint CSVs into PNG image representations suitable for use with diffusion models and other visual architectures.
4. **Take an already generated set of image-representation of images and convert them back into nprint representation for downstream task utilization**

This will help you understand why converting network traces to images can unlock generative modeling capabilities that traditional ML approaches struggle with.

Q1:
First, download and unzip the data you will need from (https://drive.google.com/file/d/1hY6nNXEYOwl1l-O_nCknO9xezcHr6ZXi/view?usp=sharing)
In your own words, describe how an nPrint CSV encodes a network trace.
Why might a 2D image representation capture structural relationships that a row-by-row CSV cannot?

<font color='#dbb1fb'> A 2D image representation can show all the information about the different protocols on one screen, unlike a CSV.

Q2. Design a method for converting nPrint representations of traces (in the folder real_nprints) into image representations.
Your image representation should use only the first 1024 packets from each trace to avoid producing images that are too large.
Save the images into a folder called './nd_data/student_converted_images'

<font color='#dbb1fb'> I would use matplotlib color mapping.

In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [6]:
#start at netflix_1024_17.png
starting_point = os.listdir("nd_data/real_nprints")[89:]

In [7]:
def nprint_to_image(file_name):
    for file in starting_point:
        file_name = (f"nd_data/real_nprints/{file}")
        csv_name = pd.read_csv(file_name)
        csv_name = csv_name.drop(columns="Unnamed: 0")

        Z = csv_name

        plt.pcolormesh(Z)
        plt.title(f"Mapping of {file}")

        plt.tight_layout()
        #plt.show()

        file = file.split(".")
        plt.savefig(f"nd_data/student_gen_images/{file[0]}.png")

In [ ]:
#get all files from the folder

for file in starting_point:
    nprint_to_image(f"nd_data/real_nprints/{file}")

KeyboardInterrupt: 

In [5]:
# The following is a pre-defined script used in NetDiffusion that will convert all of the provided real nprints into image representations. Run this code and observe the output
!python ./scripts/nprint_to_png.py -i ./nd_data/real_nprints/ -o ./nd_data/real_traffic_images

Processing amazon_1024.nprint
Processing amazon_1024_1.nprint
Processing amazon_1024_10.nprint
Processing amazon_1024_11.nprint
Processing amazon_1024_12.nprint
Processing amazon_1024_13.nprint
Processing amazon_1024_14.nprint
Processing amazon_1024_15.nprint
Processing amazon_1024_16.nprint
Processing amazon_1024_17.nprint
Processing amazon_1024_18.nprint
Processing amazon_1024_19.nprint
Processing amazon_1024_2.nprint
Processing amazon_1024_3.nprint
Processing amazon_1024_4.nprint
Processing amazon_1024_5.nprint
Processing amazon_1024_6.nprint
Processing amazon_1024_7.nprint
Processing amazon_1024_8.nprint
Processing amazon_1024_9.nprint
Processing facebook_1024.nprint
Processing facebook_1024_1.nprint
Processing facebook_1024_10.nprint
Processing facebook_1024_11.nprint
Processing facebook_1024_12.nprint
Processing facebook_1024_13.nprint
Processing facebook_1024_14.nprint
Processing facebook_1024_15.nprint
Processing facebook_1024_16.nprint
Processing facebook_1024_17.nprint
Proces

c:\Users\merin\ml-systems-copy\docs\assignments\scripts\nprint_to_png.py:26: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  np_df = np.array(df.applymap(np.array).to_numpy().tolist())
c:\Users\merin\ml-systems-copy\docs\assignments\scripts\nprint_to_png.py:26: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  np_df = np.array(df.applymap(np.array).to_numpy().tolist())
c:\Users\merin\ml-systems-copy\docs\assignments\scripts\nprint_to_png.py:26: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  np_df = np.array(df.applymap(np.array).to_numpy().tolist())
c:\Users\merin\ml-systems-copy\docs\assignments\scripts\nprint_to_png.py:26: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  np_df = np.array(df.applymap(np.array).to_numpy().tolist())
c:\Users\merin\ml-systems-copy\docs\assignments\scripts\nprint_to_png.py:26: FutureWarning: DataFrame.applymap has b

Q3: Now that you have seen how NetDiffusion converts nPrints into images, compare their method with the approach you designed in Q2. What are the advantages and disadvantages of each, especially in terms of what might help or hinder a vision model’s ability to learn?

<font color='#dbb1fb'>My method of using matplotlib color mapping was more intuitive for someone less familiar with machine learning, however it took longer to run and probably ended up less machine readable. The NetDiffusion method prioritized speed and machine intepretable instructions, but on the flip side, I don't think it was human intuitive.


---

## **Part 2 — Converting Generated Images Back Into Usable Format**

In the first part of the assignment, you explored how network traces in nPrint format can be transformed into image representations suitable for vision-based generative models. In Part 2, we focus on the reverse process: taking synthetic images produced by these models and converting them back into structured network representations.

This step is crucial because real-world applications do not operate on images—they require valid, interpretable packet traces that can be analyzed, replayed, or integrated into downstream tools.

---

### **Your Task for Part 2**

In this part of the assignment, you will:

1. **Convert generated images back into the original nPrint representation.**
   You will follow a scripted pipeline that translates pixel intensities and color channels back into binary header fields, reconstructing the packet-level structure of the trace.

2. **Apply essential post-processing techniques to correct errors introduced by diffusion models.**
   Generated images are rarely perfect—vision models may introduce color drift, pixel misalignment, noise, or structural artifacts.
   You will observe how heuristic correction, formatting enforcement, and reconstruction steps ensure that the converted nPrints become:

   * syntactically valid,
   * structurally consistent,
   * and replayable.

Across this section, your goal is to understand **why the reverse transformation is fragile**, which types of artifacts break reversibility, and how post-processing logic helps repair or compensate for generative errors.

For simplicity of this assignment, we have trainined and generated the images for you. If you have sufficient GPU access and want to try fine-tuning the model and generating the images yourself, feel free to take a look at the public repo (https://github.com/noise-lab/NetDiffusion).


Q4: We have taken the images converted by NetDiffusion and trained a LoRA-fine-tuned Stable Diffusion model (with ControlNet) to generate synthetic traffic images for you. These generated samples are stored in generated_traffic_images/.
Compare these generated images visually with the real images you saw earlier in real_traffic_images/.

Do you notice anything different between the real and generated traffic images? What immediately stands out as potentially problematic if we attempt to convert these generated images back into nPrint format? (Descriptive Only)

<font color='#dbb1fb'> They have different color mapping and the resolution is lower on the traffic generated images. It might be problematic to convert the generated traffic images to nprints because we do not know what the nprints themselves look like, so our only comparison is the real nprints, which have different images.

Q5: If you were to design a method to convert generated images back into nPrints, how would you do it? Explain your approach and describe how your method addresses the concerns you raised in Q4. (Descriptive only)

<font color='#dbb1fb'> I would convert the images to RGB then to binary. Then, I would test reverse the color map, but also figure out how the bits in nprint are formatted so then the colors go back to representing the network information.

Below are a set of pre-written scripts that perform the necessary post-generation augmentation and processing on the synthetic images you obtained from the diffusion model. These scripts handle tasks such as color normalization/augmentation and conversion from generated images back into nPrint format.
(The PCAP step is optional — you may run it if you are interested in observing or replaying the reconstructed traffic.)

In [1]:
# Step 1: Color Augmentation
!python ./scripts/color_processor.py \
  --input_dir="./nd_data/generated_traffic_images" \
  --output_dir="./nd_data/color_corrected_generated_traffic_images"

Processed 100 images.


In [2]:
# Step 2: Image-to-nPrint Conversion
!python ./scripts/image_to_nprint.py \
  --org_nprint ./scripts/column_example.nprint \
  --input_dir ./nd_data/color_corrected_generated_traffic_images \
  --output_dir ./nd_data/generated_nprint

Processing ./nd_data/color_corrected_generated_traffic_images\amazon_0.png with size 1088 x 1024
Saved ./nd_data/generated_nprint\amazon_0.nprint
Processing ./nd_data/color_corrected_generated_traffic_images\amazon_1.png with size 1088 x 1024
Saved ./nd_data/generated_nprint\amazon_1.nprint
Processing ./nd_data/color_corrected_generated_traffic_images\amazon_2.png with size 1088 x 1024
Saved ./nd_data/generated_nprint\amazon_2.nprint
Processing ./nd_data/color_corrected_generated_traffic_images\amazon_3.png with size 1088 x 1024
Saved ./nd_data/generated_nprint\amazon_3.nprint
Processing ./nd_data/color_corrected_generated_traffic_images\amazon_4.png with size 1088 x 1024
Saved ./nd_data/generated_nprint\amazon_4.nprint
Processing ./nd_data/color_corrected_generated_traffic_images\amazon_5.png with size 1088 x 1024
Saved ./nd_data/generated_nprint\amazon_5.nprint
Processing ./nd_data/color_corrected_generated_traffic_images\amazon_6.png with size 1088 x 1024
Saved ./nd_data/generated_n

Q6: You may now read through the provided scripts in color_processor.py (color augmentation / normalization) and image_to_nprint.py (image → nPrint reconstruction).
How do the post-processing methods implemented in these scripts compare to the approach you proposed in Q5?
Describe the pros and cons of both methods and highlight any differences in design philosophy, robustness, or assumptions.

<font color='#dbb1fb'> I think I was on the right path, but my response was way to high level in comparison to the actual implementation. The actual implementation was much more robust since it took into turning the ip address to binary then turning the rgb into numbers and turning the image into a dataframe. I assumed that the method I used to turn the nprints into images would be easily reversible with a built-in method, while the actual implementation relies on domain knowledge to convert the colors back into binary.


---

# **Part 3 — Using Real and Synthetic nPrints for Application Classification**

In the previous parts, you learned how network traces can be converted between nPrint and image representations, generated using diffusion models, and reconstructed back into nPrint format.
Now, you will evaluate how useful these generated nPrints are for downstream **machine learning tasks**.

Each nPrint file—whether real or generated—is labeled with the **application** that produced the traffic (e.g., `amazon_1.nprint` means this sample came from Amazon traffic).
In this section, you will treat each **entire nPrint file as a single sample** and build a simple ML pipeline to classify application labels.

To simplify the task, you will restrict your model to use **only the first 3 packets** (3 rows) from each nPrint.
This mimics “early packet classification,” where only the beginning of a flow is available.

---


Q7:

You now have access to both `real_nprints/` and `generated_nprint/`.
Notice that in both directories, files are labeled using the application associated with that nPrint (e.g., `amazon_1.nprint`).
Treat each **nPrint file** as one sample.

**Design an ML pipeline that trains a model using *synthetic nPrints* (from `generated_nprint/`) and evaluates its performance on *real nPrints* (from `real_nprints/`) to predict the correct application label.**

Your pipeline should:

1. Use only the **first 3 packets (first 3 rows)** of each nPrint file as input features.
2. Train a classifier on real data.
3. Test the classifier on generated data.
4. Report how well the classifier performs.

We have already written the script to load the real and generated nprints into DataFrame for you.

In [3]:
import os
import glob
import numpy as np
import pandas as pd

# ---------------------------------------------------------------
# Safe conversion for nPrint cell
# ---------------------------------------------------------------
def safe_convert(x):
    if pd.isna(x):
        return 0
    x = str(x).strip()

    if x in ["0", "1", "-1"]:
        return int(x)

    if all(c in "01" for c in x) and len(x) <= 16:
        return int(x, 2)

    if x.lstrip("-").isdigit():
        return int(x)

    return 0


# ---------------------------------------------------------------
# Get original column names from a reference nPrint
# ---------------------------------------------------------------
def get_original_columns(example_path="nd_data/real_nprints"):
    first_file = glob.glob(os.path.join(example_path, "*.nprint"))[0]

    df = pd.read_csv(first_file)

    # Drop index column if present (like "Unnamed: 0")
    if df.columns[0].lower().startswith("unnamed"):
        df = df.drop(df.columns[0], axis=1)

    return list(df.columns)


# ---------------------------------------------------------------
# Load nPrint → first 3 rows → flatten with prefixed column names
# ---------------------------------------------------------------
def load_nprint_with_colnames(path, base_cols, num_rows=3):
    df = pd.read_csv(path, dtype=str, low_memory=False)

    # Drop "Unnamed: 0" if present
    if df.columns[0].lower().startswith("unnamed"):
        df = df.drop(df.columns[0], axis=1)

    df = df.iloc[:num_rows, :]            # first 3 packets  
    df = df.map(safe_convert)             # clean convert  

    # Build prefixed column names
    pkt_cols = []
    for pkt in range(1, num_rows + 1):
        pkt_cols.extend([f"pkt{pkt}_{c}" for c in base_cols])

    # Flatten 3×columns into 1 vector
    flat = df.values.flatten()

    return flat, pkt_cols


# ---------------------------------------------------------------
# Load entire directory into a DataFrame (with labels)
# ---------------------------------------------------------------
def load_directory_as_df(directory, base_cols):
    rows = []
    labels = []
    colnames_set = None

    for path in glob.glob(os.path.join(directory, "*.nprint")):
        label = os.path.basename(path).split("_")[0]

        flat, cn = load_nprint_with_colnames(path, base_cols)
        rows.append(flat)
        labels.append(label)

        if colnames_set is None:
            colnames_set = cn   # only set once

    df = pd.DataFrame(rows, columns=colnames_set)
    df["label"] = labels
    return df


# ---------------------------------------------------------------
# FINAL: Load real + synthetic DataFrames
# ---------------------------------------------------------------
base_cols = get_original_columns("nd_data/real_nprints")

df_synth = load_directory_as_df("nd_data/generated_nprint", base_cols)
df_real  = load_directory_as_df("nd_data/real_nprints", base_cols)

print("Synthetic DF:", df_synth.shape)
print(df_synth.head())

print("\nReal DF:", df_real.shape)
print(df_real.head())


Synthetic DF: (100, 3265)
   pkt1_ipv4_ver_0  pkt1_ipv4_ver_1  pkt1_ipv4_ver_2  pkt1_ipv4_ver_3  \
0                0                0                0                0   
1                1                0                0                0   
2                1                0                0                0   
3                1                0                0                0   
4                1                0                0                0   

   pkt1_ipv4_hl_0  pkt1_ipv4_hl_1  pkt1_ipv4_hl_2  pkt1_ipv4_hl_3  \
0               0               0               0               0   
1               0               0               0               0   
2               0               0               0               0   
3               0               0               0               0   
4               0               0               1               0   

   pkt1_ipv4_tos_0  pkt1_ipv4_tos_1  ...  pkt3_icmp_roh_23  pkt3_icmp_roh_24  \
0                0                0  ...

1. Use only the **first 3 packets (first 3 rows)** of each nPrint file as input features.
2. Train a classifier on real data.
3. Test the classifier on generated data.
4. Report how well the classifier performs.


In [11]:
le = LabelEncoder()

In [13]:
#Use first three rows as features (real)
X = df_real[df_real.columns.values[:-1]]

y = df_real["label"]

y = le.fit_transform(y)

#Train on real data
clf = RandomForestClassifier(max_depth=10, random_state=42)
clf.fit(X, y)

RandomForestClassifier(max_depth=10, random_state=42)

In [14]:
#Declare testing (synthetic)
X_test = df_synth[df_synth.columns.values[:-1]]

y_test = df_synth["label"]

y_test = le.transform(y_test)

In [15]:
y_pred = clf.predict(X_test)

In [16]:
#Test predictions against reality
accuracy_score(y_test, y_pred)

0.3

<font color='#dbb1fb'> The classifier does not perform well when trained on real data, then tested on synthetic. I'm going to try scaling to see if it's a data distribution problem or if it's a model training problem.

In [5]:
#Use first three rows as features (real)
X = df_real[df_real.columns.values[:-1]]
X_scaled = StandardScaler().fit_transform(X)

y = df_real["label"]

y = le.fit_transform(y)

#Train on real data
clf = RandomForestClassifier(max_depth=10, random_state=42)
clf.fit(X_scaled, y)

RandomForestClassifier(max_depth=10, random_state=42)

In [6]:
#Declare testing (synthetic)
X_test = df_synth[df_synth.columns.values[:-1]]
X_test_scaled = StandardScaler().fit_transform(X_test)

y_test = df_synth["label"]

y_test = le.transform(y_test)

In [9]:
#Generate prediction
y_pred = clf.predict(X_test_scaled)

In [10]:
#Test predictions against reality
accuracy_score(y_test, y_pred)

0.28

<font color='#dbb1fb'> The classifier has lower accuracy when scaled, so might be a data problem. I'm going to try the other way of training and testing, since the instructions above the numbered list suggest training on synthetic and testing on real.

In [23]:
#Use first three rows as features (synthetic)
X = df_synth[df_synth.columns.values[:-1]]
y = df_synth["label"]

y = le.fit_transform(y)

#Train on real data
clf = RandomForestClassifier(max_depth=10, random_state=42)
clf.fit(X, y)

RandomForestClassifier(max_depth=10, random_state=42)

In [24]:
#Declare testing (real)
X_test = df_real[df_real.columns.values[:-1]]
y_test = df_real["label"]

y_test = le.transform(y_test)

In [25]:
#Generate prediction
y_pred = clf.predict(X_test)

In [26]:
#Test predictions against reality
accuracy_score(y_test, y_pred)

0.28

<font color='#dbb1fb'> The classifier has lower accuracy when trained on synthetic data, then tested on real data. Will also try the scaling this way around.

In [17]:
#Use first three rows as features (synthetic)
X = df_synth[df_synth.columns.values[:-1]]
X_scaled = StandardScaler().fit_transform(X)

y = df_synth["label"]

y = le.fit_transform(y)

#Train on real data
clf = RandomForestClassifier(max_depth=10, random_state=42)
clf.fit(X_scaled, y)

RandomForestClassifier(max_depth=10, random_state=42)

In [18]:
#Declare testing (real)
X_test = df_real[df_real.columns.values[:-1]]
X_test_scaled = StandardScaler().fit_transform(X_test)

y_test = df_real["label"]

y_test = le.transform(y_test)

In [19]:
#Generate prediction
y_pred = clf.predict(X_test_scaled)

In [20]:
#Test predictions against reality
accuracy_score(y_test, y_pred)

0.275

<font color='#dbb1fb'> The scaled training on synthetic and testing on real has the lowest accuracy.